## Etapa 2
Com base na última etapa, o modelo definido como base para fine-tuning foi o `convnext_tiny`

O transform tem como objetivo manipular as imagens que serão introduzidas no modelo para treino, o objetivo é testar diferentes combinações e recursos presentes no transform para descobrir o que melhor se encaixa para a solução.

In [10]:
import numpy as np

import torch
import torch.nn as nn

from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

from sklearn.metrics import f1_score, balanced_accuracy_score

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


### Data
Os dados são de milho, com uma boa diferenciação entre as doenças, apresentando diferentes cores, texturas e formas.

In [11]:
from dotenv import load_dotenv
import os

In [18]:
if load_dotenv():
    print("Env carregado com sucesso.") 

    DATA_DIR = os.getenv("DATA_PATH")
    BATCH_SIZE = 32

    print(DATA_DIR)
else:
    raise ImportError("Erro ao carregar env")

Env carregado com sucesso.
/media/kaua-matheus/HD320/Data/Agroscope/Especialista/Corn


### Transform
O transform padrão para o modelo ConvNext_Tiny

Para avaliação, não se deve aplicar filtragens ou manipulações nas imagens pois essas refletem as condições reais de uma imagem de entrada (vinda diretamente do produtor).
Diferente do treino, onde introduzimos o máximo de condições e variações com o objetivo de treinar o modelo com o máximo de possibilidades.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(.85, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225])
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [.485, .456, .406], 
        [.229, .224, .225]),
])

In [15]:
full_ds = ImageFolder(DATA_DIR)
num_classes = len(full_ds)

In [16]:
n = len(full_ds)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val

In [17]:
train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

In [19]:
val_ds.dataset.transform = eval_tf
test_ds.dataset.transform = eval_tf

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

In [20]:
print("Classes:", full_ds.classes)
print(f"Split -> train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

Classes: ['Healthy', 'Rust_Blight', 'Rust_Common']
Split -> train=2051 val=439 test=441


### Modelo

In [22]:
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
in_f = model.classifier[2].in_features
model.classifier[1] = nn.Linear(in_f, num_classes)

# Não será utilizado no momento
# head_params = model.classifier.parameters()
# last_block_params = model.features[-1].parameters()

In [23]:
for p in model.parameters():
        p.requires_grad = False

keywords = ["fc", "classifier", "heads", "layer4", "denseblock4", "features.7", "features.8", "encoder.layers.11"]

for n, p in model.named_parameters():
    if any(k in n for k in keywords):
        p.requires_grad = True

### Otimizadores
Por que dos otimizadores?

In [24]:
epochs = 12
lr = 1e-3
weight_decay=1e-4

In [25]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

### Treino

In [ ]:
model.to(DEVICE)

for epoch in range(epochs):

    running_loss = .0
    model.train()

    print(f"Iteration: {epoch}")
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        inputs = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        scheduler.step()


    corrects = []
    model.eval()
    classified_right = 0

    print("Evaluationg on validation set")
    for i, (images, labels) in enumerate(val_loader):
        with torch.no_grad():
            inputs = images.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = model(inputs)
            _, pred_classes = torch.max(outputs, 1)

            loss = criterion(outputs, labels)
            classified_right += (pred_classes == labels).sum().item()

    accuracy = classified_right / len(val_ds)
    print(f"Epoch {epoch} Accuracy: {accuracy:.3f}")

Iteration: 0
